## 0. 환경설정

In [1]:
!pip install neo4j-graphrag neo4j openai mysql-connector-python pandas gradio python-dotenv -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
from dotenv import load_dotenv

load_dotenv()  # .env 파일에서 환경변수 로드

import pandas as pd
import mysql.connector
import gradio as gr

from neo4j import GraphDatabase, basic_auth
from neo4j.time import Date

from neo4j_graphrag.retrievers import Text2CypherRetriever
try:
    from neo4j_graphrag.llm import OpenAILLM
except ImportError:
    from neo4j_graphrag.llm.openai_llm import OpenAILLM
from neo4j_graphrag.generation import GraphRAG
import re
import time

import requests
from math import radians, cos, sin, asin, sqrt, atan2

print("모듈 로드 완료")

c:\Users\SSAFY\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


모듈 로드 완료


In [3]:
# OPENAI_API_KEY는 .env에서 자동 로드됨
# 확인용:
print("OpenAI API Key 설정됨:", bool(os.environ.get("OPENAI_API_KEY")))

OpenAI API Key 설정됨: True


In [4]:
# KAKAO_REST_API_KEY는 .env에서 자동 로드됨
# 확인용:
print("Kakao API Key 설정됨:", bool(os.environ.get("KAKAO_REST_API_KEY")))

Kakao API Key 설정됨: True


In [5]:
api_key  = os.environ.get("KAKAO_REST_API_KEY")
print("exists:", bool(api_key))
print("repr:", repr(api_key))
print("prefix:", api_key[:8] if api_key else None)

exists: True
repr: 'aa921e29bd619d0b1ad79ece4078f34b'
prefix: aa921e29


## 0-1. MY SQL 및 NEO4J 불러오기

In [6]:
MYSQL_HOST = os.environ.get("MYSQL_HOST", "localhost")
MYSQL_USER = os.environ.get("MYSQL_USER", "")
MYSQL_PASSWORD = os.environ.get("MYSQL_PASSWORD", "")
MYSQL_DB = os.environ.get("MYSQL_DB", "")
MYSQL_PORT = int(os.environ.get("MYSQL_PORT", 3306))

In [7]:
NEO4J_URI = os.environ.get("NEO4J_URI", "neo4j://127.0.0.1:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PW", "")

In [8]:
COUPLE_ID = 14  # 실제 존재하는 커플 ID


In [9]:
# 기존 연결 닫기 (중복 연결 방지)
try:
    if conn and conn.is_connected():
        conn.close()
        print("기존 MySQL 연결 종료")
except Exception:
    pass
conn = None

if all([MYSQL_HOST, MYSQL_USER, MYSQL_PASSWORD, MYSQL_DB]):
    try:
        conn = mysql.connector.connect(
            host=MYSQL_HOST,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD,
            database=MYSQL_DB,
            port=int(MYSQL_PORT),
            ssl_disabled=False,
            connection_timeout=10
        )
        print("MySQL 연결 성공")
    except Exception as e:
        conn = None
        print(f"MySQL 연결 실패: {e}")
else:
    print("MySQL 미설정 (.env 확인 필요)")

MySQL 연결 실패: 1203 (42000): User S14P22A105 already has more than 'max_user_connections' active connections


### 0-1-1. 사용자 선호 조회

In [10]:
query = """
SELECT *
FROM COUPLE_PREFERENCES
WHERE couple_id = %s
"""

df_pref = pd.read_sql(query, conn, params=(COUPLE_ID,))

if df_pref.empty:
    print(f"couple_id={COUPLE_ID} 선호 데이터 없음 (더미 사용)")
    pref = {"couple_id": COUPLE_ID, "region": "서울", "budget": "300만원"}
else:
    pref = df_pref.iloc[0].to_dict()
    print("=== 사용자 선호 ===")
    print(pref)
    print("=== 컬럼 목록 ===")
    print(df_pref.columns.tolist())


AttributeError: 'NoneType' object has no attribute 'cursor'

### 0-1-2. 사용자 취향

In [ ]:
def get_user_preference(conn, couple_id):
    query = """
    SELECT *
    FROM couple_preferences
    WHERE couple_id = %s
    """
    df = pd.read_sql(query, conn, params=(couple_id,))

    if df.empty:
        return None

    return df.iloc[0].to_dict()


### 0-1-3. 좋아요

In [ ]:
def get_user_likes(conn, couple_id):
    # couple_venue_likes 테이블이 없으므로 빈 리스트 반환
    return []


### 0-1-4. 좋아요 누른 홀이름

In [ ]:
def get_liked_halls_with_names(conn, couple_id):
    # couple_venue_likes 테이블이 없으므로 빈 리스트 반환
    return []


### 0-1-5. 좋아요 기반 추천(+ 평점 순)

In [ ]:
def recommend_by_likes(conn, couple_id):
    """
    couple_venue_likes 테이블이 없으므로,
    couple_preferences 기반으로 vendor_hall_detail 매칭 추천 반환.
    """
    pref = get_user_preference(conn, couple_id)
    if not pref:
        return []

    style = pref.get("style") or ""
    guest_count_raw = pref.get("guest_count") or ""
    budget_raw = pref.get("budget") or ""

    # guest_count 파싱 (예: "200명", "100~150명")
    import re
    guest_nums = [int(x) for x in re.findall(r"\d+", guest_count_raw)]
    guest_min = guest_nums[0] if guest_nums else 0
    guest_max = guest_nums[-1] if len(guest_nums) > 1 else guest_min

    # budget 파싱 (만원 단위 → 원)
    budget_nums = [int(x) for x in re.findall(r"\d+", budget_raw)]
    budget_val = budget_nums[0] * 10000 if budget_nums else None

    query = """
    SELECT
        v.id AS vendor_id,
        v.name,
        v.rating,
        v.review_count AS reviewCnt,
        vhd.style,
        vhd.guest_min,
        vhd.guest_max,
        vhd.meal_type,
        vhd.has_subway,
        vhd.has_parking,
        vp.price
    FROM vendor v
    JOIN vendor_hall_detail vhd ON v.id = vhd.vendor_id
    LEFT JOIN vendor_package vp ON v.id = vp.vendor_id
    WHERE v.category = '웨딩홀'
      AND (%s = '' OR vhd.style = %s)
      AND (%s = 0 OR vhd.guest_min <= %s)
      AND (%s = 0 OR vhd.guest_max >= %s)
    GROUP BY v.id, v.name, v.rating, v.review_count,
             vhd.style, vhd.guest_min, vhd.guest_max, vhd.meal_type,
             vhd.has_subway, vhd.has_parking, vp.price
    ORDER BY v.rating DESC, v.review_count DESC
    LIMIT 5
    """

    params = (style, style, guest_min, guest_min, guest_max if guest_max else 9999, guest_max if guest_max else 9999)
    df = pd.read_sql(query, conn, params=params)
    return df.to_dict(orient="records")


In [ ]:
def recommend_by_likes_with_region_boost(conn, couple_id, preferred_region=None, preferred_sub_region=None):
    """
    couple_preferences + vendor + vendor_hall_detail 기반 추천.
    vendor 테이블에는 region 컬럼이 없으므로 description/hashtags로 지역 필터.
    """
    pref = get_user_preference(conn, couple_id)
    if not pref:
        return []

    style = pref.get("style") or ""
    guest_count_raw = pref.get("guest_count") or ""

    import re
    guest_nums = [int(x) for x in re.findall(r"\d+", guest_count_raw)]
    guest_min = guest_nums[0] if guest_nums else 0
    guest_max = guest_nums[-1] if len(guest_nums) > 1 else (guest_min if guest_min else 9999)

    region_filter = ""
    params = [style, style, guest_min, guest_min, guest_max, guest_max]

    if preferred_region:
        region_filter = "AND (v.description LIKE %s OR v.hashtags LIKE %s)"
        params += [f"%{preferred_region}%", f"%{preferred_region}%"]

    query = f"""
    SELECT
        v.id AS vendor_id,
        v.name,
        v.rating,
        v.review_count AS reviewCnt,
        vhd.style,
        vhd.guest_min,
        vhd.guest_max,
        vhd.meal_type,
        vhd.has_subway,
        vhd.has_parking,
        vp.price
    FROM vendor v
    JOIN vendor_hall_detail vhd ON v.id = vhd.vendor_id
    LEFT JOIN vendor_package vp ON v.id = vp.vendor_id
    WHERE v.category = '웨딩홀'
      AND (%s = '' OR vhd.style = %s)
      AND (%s = 0 OR vhd.guest_min <= %s)
      AND (%s = 0 OR vhd.guest_max >= %s)
      {region_filter}
    GROUP BY v.id, v.name, v.rating, v.review_count,
             vhd.style, vhd.guest_min, vhd.guest_max, vhd.meal_type,
             vhd.has_subway, vhd.has_parking, vp.price
    ORDER BY v.rating DESC, v.review_count DESC
    LIMIT 5
    """

    df = pd.read_sql(query, conn, params=params)
    return df.to_dict(orient="records")


## NEO4J 연결

In [ ]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=basic_auth(NEO4J_USER, NEO4J_PASSWORD)
)

In [ ]:
with driver.session() as session:
    count_result = session.run("MATCH (n) RETURN count(n) AS cnt")
    print("Neo4j node count:", count_result.single()["cnt"])

Neo4j node count: 4667


## 스키마 함수 추출

In [ ]:
def get_node_datatype(value):
    if isinstance(value, str):
        return "STRING"
    elif isinstance(value, int):
        return "INTEGER"
    elif isinstance(value, float):
        return "FLOAT"
    elif isinstance(value, bool):
        return "BOOLEAN"
    elif isinstance(value, list):
        return f"LIST[{get_node_datatype(value[0])}]" if value else "LIST"
    elif isinstance(value, Date):
        return "DATE"
    else:
        return "UNKNOWN"

In [ ]:
def get_schema(uri, user, password):
    driver_local = GraphDatabase.driver(uri, auth=basic_auth(user, password))

    with driver_local.session() as session:
        node_query = """
        MATCH (n)
        WITH DISTINCT labels(n) AS node_labels, keys(n) AS property_keys, n
        UNWIND node_labels AS label
        UNWIND property_keys AS key
        RETURN label, key, n[key] AS sample_value
        """

        rel_query = """
        MATCH ()-[r]->()
        WITH DISTINCT type(r) AS rel_type, keys(r) AS property_keys, r
        UNWIND property_keys AS key
        RETURN rel_type, key, r[key] AS sample_value
        """

        rel_direction_query = """
        MATCH (a)-[r]->(b)
        RETURN DISTINCT labels(a) AS start_label, type(r) AS rel_type, labels(b) AS end_label
        ORDER BY start_label, rel_type, end_label
        """

        nodes = session.run(node_query)
        relationships = session.run(rel_query)
        rel_directions = session.run(rel_direction_query)

        schema = {
            "nodes": {},
            "relationships": {},
            "relations": []
        }

        for record in nodes:
            label = record["label"]
            key = record["key"]
            sample_value = record["sample_value"]
            inferred_type = get_node_datatype(sample_value)

            if label not in schema["nodes"]:
                schema["nodes"][label] = {}
            schema["nodes"][label][key] = inferred_type

        for record in relationships:
            rel_type = record["rel_type"]
            key = record["key"]
            sample_value = record["sample_value"]
            inferred_type = get_node_datatype(sample_value)

            if rel_type not in schema["relationships"]:
                schema["relationships"][rel_type] = {}
            schema["relationships"][rel_type][key] = inferred_type

        for record in rel_directions:
            start_labels = record["start_label"]
            end_labels = record["end_label"]

            if not start_labels or not end_labels:
                continue

            start_label = start_labels[0]
            rel_type = record["rel_type"]
            end_label = end_labels[0]

            schema["relations"].append(f"(:{start_label})-[:{rel_type}]->(:{end_label})")

    driver_local.close()
    return schema

In [ ]:
def format_schema(schema):
    result = []
    result.append("Node properties:")
    for label, properties in schema["nodes"].items():
        props = ", ".join(f"{k}: {v}" for k, v in properties.items())
        result.append(f"{label} {{{props}}}")
    result.append("Relationship properties:")
    for rel_type, properties in schema["relationships"].items():
        props = ", ".join(f"{k}: {v}" for k, v in properties.items())
        result.append(f"{rel_type} {{{props}}}")
    result.append("The relationships:")
    for relation in schema["relations"]:
        result.append(relation)
    return "\n".join(result)


schema = get_schema(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
base_schema = format_schema(schema)

# 실제 데이터 예시를 포함한 enriched schema
ENRICHED_SCHEMA = base_schema + """

=== 실제 저장된 값 예시 (Cypher 생성 시 반드시 참고) ===

[Hall 노드 주요 속성]
- name: 홀 이름 전체 (예: '아모리스 역삼점', '더채플앳청담', '엘타워', 'JW Marriott 서울_서초')
  * 이름에 지역명이 포함된 경우 많음 (역삼, 청담, 강남, 서초 등)
- region: 광역 지역 (예: '서울', '경기', '부산', '서울 강남구', '경기 성남시')
  * 주의: '서울특별시' 가 아닌 '서울' 로 저장됨
- subRegion: 구/시 단위 (예: '강남구', '서초구', '마포구', '파주시')
- address: 주소 (예: '서울 강남구', '서울 서초구', '경기 성남시')
  * 상세 도로명 없이 시/구 수준으로만 저장된 경우 많음
- rating: 평점 0~100 (예: 82.0, 95.0)
- memoContent: 홀 상세 설명 텍스트
  * 예: "스타일: 어두운 / 컨벤션", "역삼역 도보 3분", "야외 정원 보유"
  * 특징/분위기 검색 시 CONTAINS 사용
- minIndividualHallPrice / maxIndividualHallPrice: 홀 전체 대관료 (원 단위)
  * '대관료', '홀대관료', '대관 비용' 등 사용자가 대관료를 물으면 반드시 minIndividualHallPrice 사용
- minRentalPrice / maxRentalPrice: 예식 렌탈료 (홀 대관료와 별개 항목)
- minMealPrice / maxMealPrice: 식대 (원 단위)
- memoContent: 최대 수용인원, 보증인원, 스타일, 예식형태 등 텍스트로 포함
  * 예: '최대 수용인원: 400명', '보증인원: 300명', '역삼역 도보 3분'

[Region 노드]
- name 예시: '서울', '경기', '부산', '대구', '서울 강남구', '서울 서초구', '경기 성남시'
  * 광역시/도 단위이거나 '서울 강남구' 처럼 구 단위까지 포함된 경우도 있음

[District 노드]
- name 예시: '강남구', '서초구', '마포구', '종로구', '파주시', '고양시'
  * 구(區) 또는 시(市) 단위

[StyleFilter 노드]
- name 전체 목록: '우아한', '심플한', '러블리', '깨끗/화사', '음영', '내추럴', '인물중심', '인물+배경', '빈티지', '클래식', '할인'

=== 위치 검색 규칙 ===
1. "역삼", "청담", "홍대", "합정" 같은 동/역 이름
   → Region이나 District에 없음 → h.name CONTAINS '역삼' OR h.address CONTAINS '역삼' 사용
2. "강남구", "서초구" 같은 구 단위
   → MATCH (h:Hall)-[:IN_DISTRICT]->(d:District) WHERE d.name CONTAINS '강남'
   → 또는 h.subRegion CONTAINS '강남'
3. "서울", "부산" 같은 광역 단위
   → h.region CONTAINS '서울' 또는 MATCH (h:Hall)-[:IN_REGION]->(r:Region) WHERE r.name CONTAINS '서울'
4. 특성/분위기 검색 (야외, 정원, 뷔페, 코스, 모던 등)
   → h.memoContent CONTAINS '야외'
"""

neo4j_schema = ENRICHED_SCHEMA

print("=== Neo4j Schema (Enriched) ===")
print(neo4j_schema[:500])
print("...")


=== Neo4j Schema (Enriched) ===
Node properties:
Image {url: STRING, title: STRING}
Tag {name: STRING, typeName: STRING, type: INTEGER, category: STRING}
Hall {partnerId: INTEGER, name: STRING, address: STRING, profileUrl: STRING, rating: FLOAT, region: STRING, tel: STRING, typeName: STRING, uuid: STRING, minRentalPrice: INTEGER, subRegion: STRING, partnerProfileId: INTEGER, modTsp: STRING, minMealPrice: INTEGER, partnerProfileName: STRING, minIndividualHallPrice: INTEGER, maxRentalPrice: INTEGER, bookingState: INTEGER, maxMea
...


## 선호도 기반 추천 테스트

In [ ]:
cypher_query = """
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (h)-[:IN_REGION]->(r:Region)

WITH h,
     collect(DISTINCT sf.name) AS styles,
     collect(DISTINCT t.name) AS tags,
     collect(DISTINCT r.name) AS regions

WITH h, styles, tags, regions,
     CASE WHEN any(x IN styles WHERE x CONTAINS $style) THEN 2 ELSE 0 END +
     CASE WHEN any(x IN tags WHERE x CONTAINS $mood) OR h.memoContent CONTAINS $mood THEN 1 ELSE 0 END +
     CASE WHEN any(x IN regions WHERE x CONTAINS $region) THEN 2 ELSE 0 END +
     CASE WHEN any(x IN tags WHERE x CONTAINS $food) THEN 1 ELSE 0 END
     AS score

WHERE score > 0

RETURN h.partnerId AS hallId,
       h.name AS name,
       h.rating AS rating,
       h.reviewCnt AS reviewCnt,
       score
ORDER BY score DESC, rating DESC, reviewCnt DESC
LIMIT 10
"""

with driver.session() as session:
    result = session.run(
        cypher_query,
        style=str(pref.get("style", "")),
        mood=str(pref.get("mood", "")),
        food=str(pref.get("food", "")),
        region=str(pref.get("venue", pref.get("region", "")))
    )
    halls = [record.data() for record in result]

print("=== 선호 기반 추천 결과 ===")
print(halls)

=== 선호 기반 추천 결과 ===
[{'hallId': 1182, 'name': 'KDW웨딩', 'rating': 90.0, 'reviewCnt': 606, 'score': 3}, {'hallId': 12493, 'name': 'HW컨벤션센터', 'rating': 90.0, 'reviewCnt': 17, 'score': 3}, {'hallId': 11958, 'name': '메리스에이프럴', 'rating': 88.0, 'reviewCnt': 244, 'score': 3}, {'hallId': 1283, 'name': '엘컨벤션', 'rating': 87.0, 'reviewCnt': 59, 'score': 3}, {'hallId': 2482, 'name': '포레스트원 웨딩', 'rating': 86.0, 'reviewCnt': 115, 'score': 3}, {'hallId': 12113, 'name': '웨스턴베니비스 영등포점_영등포', 'rating': 85.0, 'reviewCnt': 51, 'score': 3}, {'hallId': 9652, 'name': 'BNT CONVENTION 웨딩', 'rating': 85.0, 'reviewCnt': 42, 'score': 3}, {'hallId': 3634, 'name': '로프트가든344', 'rating': 84.0, 'reviewCnt': 777, 'score': 3}, {'hallId': 1251, 'name': '웨딩여율리', 'rating': 84.0, 'reviewCnt': 619, 'score': 3}, {'hallId': 1498, 'name': '보테가마지오_성동', 'rating': 83.0, 'reviewCnt': 151, 'score': 3}]


### 예시

In [ ]:
examples = [
    """USER INPUT: '강남구에 있는 결혼식장 추천해줄래'
QUERY:
MATCH (h:Hall)-[:IN_DISTRICT]->(d:District)
WHERE d.name CONTAINS '강남'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, d.name AS district
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '서울 강남 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)-[:IN_DISTRICT]->(d:District)
WHERE d.name CONTAINS '강남'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, d.name AS district
ORDER BY h.reviewCnt DESC, h.rating DESC
LIMIT 10
""",

    """USER INPUT: '서울 종로구 결혼식장 추천'
QUERY:
MATCH (h:Hall)-[:IN_DISTRICT]->(d:District)
WHERE d.name CONTAINS '종로'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, d.name AS district
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '호텔 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
WHERE t.name CONTAINS '호텔' OR sf.name CONTAINS '호텔' OR h.typeName CONTAINS '호텔'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.rating IS NOT NULL
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '럭셔리한 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
WHERE t.name CONTAINS '럭셔리' OR sf.name CONTAINS '럭셔리'
   OR t.name CONTAINS '화이트' OR sf.name CONTAINS '화이트'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '야외 정원이 있는 웨딩홀 알려줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '야외' OR t.name CONTAINS '정원' OR h.memoContent CONTAINS '야외'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '평점 높은 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.rating IS NOT NULL AND h.rating > 0
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region
ORDER BY h.rating DESC
LIMIT 10
""",

    """USER INPUT: '노보텔 앰버서더와 비슷한 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:HAS_TAG]->(t:Tag)
WHERE t.name CONTAINS '호텔' OR t.name CONTAINS '럭셔리' OR h.typeName CONTAINS '호텔'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '부산 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.region CONTAINS '부산'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '리뷰 많은 웨딩홀 5개 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.reviewCnt IS NOT NULL AND h.reviewCnt > 0
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region
ORDER BY h.reviewCnt DESC
LIMIT 5
""",

    """USER INPUT: '역삼 근처 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.name CONTAINS '역삼'
   OR h.subRegion CONTAINS '역삼'
   OR h.address CONTAINS '역삼'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region, h.address AS address
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '홍대 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.name CONTAINS '홍대'
   OR h.subRegion CONTAINS '홍대'
   OR h.subRegion CONTAINS '마포'
   OR h.address CONTAINS '홍대'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",

    """USER INPUT: '강남역 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:IN_DISTRICT]->(d:District)
WHERE h.name CONTAINS '강남'
   OR h.address CONTAINS '강남'
   OR (d IS NOT NULL AND d.name CONTAINS '강남')
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.address AS address
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
""",


    """USER INPUT: '강남에 자연채광 있는 모던한 웨딩홀 추천해줘'
QUERY:
MATCH (h:Hall)
OPTIONAL MATCH (h)-[:IN_DISTRICT]->(d:District)
WHERE (d IS NULL OR d.name CONTAINS '강남')
  AND (h.memoContent CONTAINS '자연채광' OR h.memoContent CONTAINS '모던' OR h.memoContent CONTAINS '밝은')
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.subRegion AS subRegion, h.memoContent AS memo
ORDER BY h.rating DESC
LIMIT 10
""",

    """USER INPUT: '인터컨티넨탈 웨딩홀 알려줘'
QUERY:
MATCH (h:Hall)
WHERE h.name CONTAINS '컨티넨탈' OR h.name CONTAINS '컨티넨털' OR h.name CONTAINS 'Continental' OR h.name CONTAINS '인터컨'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.address AS address
LIMIT 10
""",

    """USER INPUT: '야외 정원 웨딩 홀 추천해줘'
QUERY:
MATCH (h:Hall)
WHERE h.memoContent CONTAINS '야외' OR h.memoContent CONTAINS '정원' OR h.memoContent CONTAINS '가든'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, h.region AS region, h.memoContent AS memo
ORDER BY h.rating DESC
LIMIT 10
""",

    """USER INPUT: '엘타워 대관료 알려줘'
QUERY:
MATCH (h:Hall)
WHERE h.name CONTAINS '엘타워'
RETURN h.name AS name, h.minIndividualHallPrice AS minHallPrice,
       h.maxIndividualHallPrice AS maxHallPrice,
       h.minMealPrice AS minMealPrice, h.maxMealPrice AS maxMealPrice
LIMIT 1
""",

    """USER INPUT: '아펠가모 반포점 수용인원 알려줘'
QUERY:
MATCH (h:Hall)
WHERE h.name CONTAINS '아펠가모 반포'
RETURN h.name AS name, h.memoContent AS memoContent
LIMIT 1
""",
]


In [ ]:
llm = OpenAILLM(
    model_name="gpt-4o",
    model_params={"temperature": 0}
)

In [ ]:
def recommend_by_preference(driver, pref):
    region = str(pref.get("venue") or pref.get("region") or "")
    style = str(pref.get("style") or "")
    mood_raw = str(pref.get("mood") or "")
    food = str(pref.get("food") or "")
    budget_raw = str(pref.get("budget") or "")
    guest_raw = str(pref.get("guest_count") or "")

    # 분위기 여러 개 분리 (콤마 구분)
    moods = [m.strip() for m in mood_raw.split(",") if m.strip()]

    # 예산 파싱 (예: "300만원" → 3000000)
    import re
    budget_nums = [int(x) for x in re.findall(r"\d+", budget_raw)]
    max_meal_budget = budget_nums[0] * 10000 if budget_nums else None

    # 게스트 수 파싱
    guest_nums = [int(x) for x in re.findall(r"\d+", guest_raw)]
    guest_count = guest_nums[0] if guest_nums else None

    cypher_query = """
    MATCH (h:Hall)

    WITH h,
         CASE WHEN $region <> '' AND (h.region CONTAINS $region OR h.address CONTAINS $region OR h.subRegion CONTAINS $region) THEN 3 ELSE 0 END AS region_score,
         CASE WHEN $style <> '' AND h.memoContent CONTAINS $style THEN 2 ELSE 0 END AS style_score,
         CASE WHEN $food <> '' AND (h.memoContent CONTAINS $food) THEN 1 ELSE 0 END AS food_score,
         CASE WHEN $guest_count IS NULL OR (h.memoContent IS NOT NULL) THEN 0 ELSE 0 END AS guest_score

    WITH h, (region_score + style_score + food_score) AS score

    WHERE ($region = '' OR h.region CONTAINS $region OR h.address CONTAINS $region)

    RETURN h.partnerId AS hallId,
           h.name AS name,
           h.rating AS rating,
           h.reviewCnt AS reviewCnt,
           h.region AS region,
           h.subRegion AS subRegion,
           h.minMealPrice AS minMealPrice,
           h.minIndividualHallPrice AS minHallPrice,
           score
    ORDER BY score DESC, h.rating DESC, h.reviewCnt DESC
    LIMIT 10
    """

    with driver.session() as session:
        result = session.run(
            cypher_query,
            region=region,
            style=style,
            food=food,
            guest_count=guest_count,
        )
        rows = [record.data() for record in result]

    # 분위기 키워드 보너스 점수 (Python에서 처리)
    if moods:
        for row in rows:
            memo = row.get("memoContent") or ""
            bonus = sum(1 for m in moods if m in memo)
            row["score"] = (row.get("score") or 0) + bonus
        rows.sort(key=lambda x: (-x.get("score", 0), -(x.get("rating") or 0), -(x.get("reviewCnt") or 0)))

    return rows


In [ ]:
def recommend_by_facility(driver, keyword, region=None):

    cypher_query = """
    MATCH (h:Hall)-[:IN_REGION]->(r:Region)

    WHERE h.memoContent CONTAINS $keyword
      AND ($region IS NULL OR r.name CONTAINS $region)

    RETURN h.partnerId AS hallId,
           h.name AS name,
           r.name AS region,
           h.rating AS rating,
           h.reviewCnt AS reviewCnt

    ORDER BY rating DESC, reviewCnt DESC
    LIMIT 10
    """

    with driver.session() as session:
        result = session.run(
            cypher_query,
            keyword=keyword,
            region=region
        )

        return [record.data() for record in result]

In [ ]:
def extract_region(message: str):
    msg = message.strip()

    region_patterns = [
        r"(서울|경기|인천|부산|대구|대전|광주|울산|세종|제주)\s*(지역|쪽)?",
    ]

    for pattern in region_patterns:
        m = re.search(pattern, msg)
        if m:
            return m.group(1)

    return None


def extract_districts(message: str):
    msg = (message or '').strip()
    found = re.findall(r'([가-힣]+(?:구|군|시))', msg)
    districts = []
    skip = {'서울시', '부산시', '대구시', '인천시', '광주시', '대전시', '울산시', '세종시'}
    for item in found:
        if item in skip:
            continue
        if item not in districts:
            districts.append(item)
    return districts

In [ ]:
def extract_facility_keyword(message: str) -> str:
    if "주차" in message or "주차장" in message:
        return "주차"
    if "발렛" in message or "발렛파킹" in message:
        return "발렛"
    if "역세권" in message:
        return "역"
    if "보증인원" in message:
        return "보증인원"
    if "코스" in message:
        return "코스"
    if "뷔페" in message:
        return "뷔페"
    return ""

In [ ]:
def kakao_search_keyword(query: str):
    import os
    import requests

    api_key = os.environ.get("KAKAO_REST_API_KEY")
    if not api_key:
        raise ValueError("KAKAO_REST_API_KEY가 설정되지 않았습니다.")

    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    headers = {"Authorization": f"KakaoAK {api_key.strip()}"}
    params = {"query": query}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
    except requests.RequestException as e:
        print(f"[KAKAO keyword 요청 실패] query={query}, error={e}")
        return None

    if resp.status_code == 429:
        print(f"[KAKAO keyword 429] query={query}")
        return None

    if resp.status_code != 200:
        print(f"[KAKAO keyword 오류] status={resp.status_code}, query={query}")
        print(resp.text)
        return None

    data = resp.json()
    docs = data.get("documents", [])
    if not docs:
        return None

    first = docs[0]
    return {
        "x": float(first["x"]),
        "y": float(first["y"]),
        "place_name": first.get("place_name"),
        "address_name": first.get("address_name"),
    }

In [ ]:
def kakao_search_address(query: str):
    import os
    import requests

    api_key = os.environ.get("KAKAO_REST_API_KEY")
    if not api_key:
        raise ValueError("KAKAO_REST_API_KEY가 설정되지 않았습니다.")

    url = "https://dapi.kakao.com/v2/local/search/address.json"
    headers = {"Authorization": f"KakaoAK {api_key.strip()}"}
    params = {"query": query}

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=10)
    except requests.RequestException as e:
        print(f"[KAKAO address 요청 실패] query={query}, error={e}")
        return None

    if resp.status_code == 429:
        print(f"[KAKAO address 429] query={query}")
        return None

    if resp.status_code != 200:
        print(f"[KAKAO address 오류] status={resp.status_code}, query={query}")
        print(resp.text)
        return None

    data = resp.json()
    docs = data.get("documents", [])
    if not docs:
        return None

    first = docs[0]
    return {
        "x": float(first["x"]),
        "y": float(first["y"]),
        "address_name": first.get("address_name"),
    }

In [ ]:
geo_cache = {}

def geocode_place(query: str):
    query = (query or "").strip()
    if not query:
        return None

    if query in geo_cache:
        return geo_cache[query]

    result = kakao_search_address(query)
    if result:
        coord = (result["y"], result["x"])
        geo_cache[query] = coord
        return coord

    result = kakao_search_keyword(query)
    if result:
        coord = (result["y"], result["x"])
        geo_cache[query] = coord
        return coord

    geo_cache[query] = None
    return None

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)

    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

In [ ]:
def search_halls_for_location(driver, region=None, districts=None, keyword=None, limit=500):
    query = """MATCH (h:Hall)
WHERE ($region IS NULL
       OR h.region = $region
       OR h.address STARTS WITH $region)
  AND (
        size($districts) = 0
        OR any(x IN $districts WHERE h.address CONTAINS x OR h.subRegion CONTAINS x)
  )
RETURN DISTINCT
       h.partnerId AS hallId,
       h.name AS name,
       h.address AS address,
       h.region AS region,
       h.subRegion AS district,
       h.rating AS rating,
       h.reviewCnt AS reviewCnt,
       h.minIndividualHallPrice AS minHallPrice,
       h.maxIndividualHallPrice AS maxHallPrice,
       h.minMealPrice AS minMealPrice,
       h.maxMealPrice AS maxMealPrice,
       CASE WHEN $keyword IS NOT NULL AND (h.name CONTAINS $keyword OR h.subRegion CONTAINS $keyword)
            THEN true ELSE false END AS name_match
ORDER BY name_match DESC
LIMIT $limit"""

    with driver.session() as session:
        result = session.run(
            query,
            region=region,
            districts=districts or [],
            keyword=keyword,
            limit=limit
        )
        rows = [record.data() for record in result]

    # 파라미터 치환한 표시용 쿼리 생성
    display_query = query
    display_query += f"""
// 파라미터: region={repr(region)}, districts={districts or []}, keyword={repr(keyword)}, limit={limit}"""

    return rows, display_query


In [ ]:
def filter_halls_by_region_and_district(halls, region=None, districts=None):
    filtered = []

    for h in halls:
        address = (h.get("address") or "").strip()
        district = (h.get("district") or "").strip()
        hall_region = (h.get("region") or "").strip()

        if region:
            if not (address.startswith(region) or hall_region == region):
                continue

        if districts:
            if not any(d in address or d in district for d in districts):
                continue

        filtered.append(h)

    return filtered

In [ ]:
def infer_region_from_start_location(start_location: str):
    if not start_location:
        return None, []

    if any(x in start_location for x in ["역삼", "강남", "선릉", "삼성", "청담", "압구정", "잠실"]):
        return "서울", []

    if any(x in start_location for x in ["판교", "분당", "수원", "용인"]):
        return "경기", []

    return None, []

In [ ]:
def calculate_distance_and_sort(start_location: str, halls: list[dict]):
    start_coord = geocode_place(start_location)
    if not start_coord:
        return []

    start_lat, start_lng = start_coord

    # 1. 구별 주소 캐싱 - distinct address만 geocode (API 호출 최소화)
    address_cache = {}
    for hall in halls:
        addr = hall.get("address") or ""
        if addr and addr not in address_cache:
            coord = geocode_place(addr)
            address_cache[addr] = coord

    # 2. name_match 홀은 이름으로 정밀 geocode (소수만)
    name_coord_cache = {}
    for hall in halls:
        if hall.get("name_match"):
            name = hall.get("name", "")
            if name and name not in name_coord_cache:
                coord = geocode_place(name)
                name_coord_cache[name] = coord

    ranked = []
    for hall in halls:
        # name_match 홀: 이름 좌표 우선
        hall_coord = None
        if hall.get("name_match"):
            hall_coord = name_coord_cache.get(hall.get("name", ""))
        # 그 외: 캐싱된 주소 좌표 사용
        if not hall_coord:
            hall_coord = address_cache.get(hall.get("address") or "")
        if not hall_coord:
            continue

        hall_lat, hall_lng = hall_coord
        distance_km = haversine_distance(start_lat, start_lng, hall_lat, hall_lng)

        item = dict(hall)
        # name_match 홀이고 이름으로 정밀 좌표를 얻었으면 해당 거리 사용
        # name_match지만 이름 geocode 실패 시 주소 거리에 -0.1 보정
        if hall.get("name_match"):
            if name_coord_cache.get(hall.get("name", "")):
                item["distance_km"] = round(distance_km, 2)
            else:
                item["distance_km"] = round(max(0.01, distance_km - 0.5), 2)
        else:
            item["distance_km"] = round(distance_km, 2)

        ranked.append(item)

    ranked.sort(
        key=lambda x: (
            x["distance_km"],
            -(x.get("rating") or 0),
            -(x.get("reviewCnt") or 0)
        )
    )
    return ranked


In [ ]:
def extract_start_location(message: str):
    msg = message.strip()

    patterns = [
        r"(.+?)에서\s*(가까운|근처|인근|몇\s*분|도보|차로|이내)",
        r"(.+?)기준으로",
        r"(.+?)\s*근처(?:로)?",
        r"(.+?)\s*인근",
        r"(.+?)\s*주변",
        r"(.+?)\s*가까운\s*곳",
    ]

    for pattern in patterns:
        m = re.search(pattern, msg)
        if m:
            location = m.group(1).strip()

            location = re.sub(
                r"(웨딩홀|예식장|추천|알려줘|찾아줘)$",
                "",
                location
            ).strip()

            if len(location) < 2:
                return None

            return location

    return None

In [ ]:
def extract_location_with_llm(message: str) -> str | None:
    # ① Regex 우선 처리: LLM 호출 없이 즉시 추출
    regex_patterns = [
        r"(.+?)에서\s*(가까운|근처|인근|몇\s*분|도보|차로|이내)",
        r"(.+?)기준으로",
        r"(.+?)\s*근처(?:로)?",
        r"(.+?)\s*인근",
        r"(.+?)\s*주변",
        r"(.+?)\s*가까운\s*곳",
    ]
    noise = re.compile(r"(웨딩홀|예식장|추천|알려줘|찾아줘)$")
    for pattern in regex_patterns:
        m = re.search(pattern, message.strip())
        if m:
            loc = noise.sub("", m.group(1)).strip()
            if len(loc) >= 2:
                return loc

    # ② Regex 실패 시에만 LLM 호출
    prompt = f"""
사용자 질문에서 출발 위치만 추출하세요.

규칙:
- 역, 지역, 동네, 랜드마크 등 위치만 추출
- 없으면 null 반환
- 설명하지 말고 JSON만 출력

예시

입력: 역삼역에서 가까운 웨딩홀
출력: {{"location": "역삼역"}}

입력: 강남 근처 웨딩홀
출력: {{"location": "강남"}}

입력: 서울에서 유명한 웨딩홀
출력: {{"location": "서울"}}

사용자 질문:
{message}
"""

    try:
        result = llm.invoke(prompt).content.strip()

        import json
        parsed = json.loads(result)

        location = parsed.get("location")

        if location and location.lower() != "null":
            return location.strip()

        return None

    except Exception as e:
        print("location extract error:", e)
        return None

In [ ]:
def rerank_location_candidates_with_llm(message: str, halls: list[dict]) -> list[dict]:
    if not halls:
        return []

    hall_lines = []
    for idx, h in enumerate(halls, start=1):
        hall_lines.append(
            f"{idx}. 이름: {h.get('name', '')}, "
            f"주소: {h.get('address', '')}, "
            f"지역: {h.get('region', '')}, "
            f"구: {h.get('district', '')}, "
            f"평점: {h.get('rating', 0)}, "
            f"리뷰수: {h.get('reviewCnt', 0)}"
        )

    prompt = f"""
당신은 웨딩홀 추천 보조 시스템입니다.

사용자 질문:
{message}

후보 웨딩홀 목록:
{chr(10).join(hall_lines)}

작업:
1. 사용자 질문에 맞지 않는 후보는 제외하세요.
2. 위치 질문이면 같은 권역의 후보를 우선하세요.
3. 질문과 무관한 지역은 제외하세요.
4. 결과는 가장 적절한 순서대로 최대 5개만 고르세요.
5. JSON만 출력하세요.

형식:
{{
  "selected_indices": [1, 3, 5]
}}
"""
    try:
        result = llm.invoke(prompt).content.strip()
        parsed = json.loads(result)
        selected = parsed.get("selected_indices", [])

        reranked = []
        for i in selected:
            if 1 <= i <= len(halls):
                reranked.append(halls[i - 1])

        return reranked if reranked else halls[:5]

    except Exception as e:
        print("LLM rerank error:", e)
        return halls[:5]

In [ ]:
def recommend_by_location(driver, conn, couple_id, message, start_location=None, count=5):
    # ② LLM 통합: 외부에서 이미 추출한 위치가 있으면 재사용 (LLM 중복 호출 제거)
    if not start_location:
        start_location = extract_location_with_llm(message)
    region = extract_region(message)
    districts = extract_districts(message)

    if not start_location or len(start_location) < 2:
        return {
            "error": "출발 위치를 이해하지 못했습니다. 예: 역삼역에서 가까운 웨딩홀 추천해줘"
        }

    if not region:
        inferred_region, inferred_districts = infer_region_from_start_location(start_location)
        region = inferred_region
        districts = districts or inferred_districts

    limit = 80 if districts else 120
    halls, neo4j_query = search_halls_for_location(driver, region=region, districts=districts, keyword=start_location, limit=limit)
    halls = filter_halls_by_region_and_district(halls, region=region, districts=districts)
    if len(halls) > max(count * 8, 40):
        halls = sorted(halls, key=lambda x: (-(x.get('name_match') or False), -(x.get('rating') or 0), -(x.get('reviewCnt') or 0)))[:max(count * 8, 40)]

    if not halls:
        return {
            "error": "조건에 맞는 웨딩홀을 찾지 못했습니다."
        }

    ranked = calculate_distance_and_sort(start_location, halls)

    if ranked:
        return {
            "mode": "distance",
            "start_location": start_location,
            "region": region,
            "results": ranked[:count],
            "neo4j_query": neo4j_query,
        }

    llm_ranked = rerank_location_candidates_with_llm(message, halls)

    return {
        "mode": "llm_fallback",
        "start_location": start_location,
        "region": region,
        "results": llm_ranked[:count],
        "neo4j_query": neo4j_query,
    }

In [ ]:
print(geocode_place("역삼역"))
print(geocode_place("서울 강남구 테헤란로"))

(37.5006744185994, 127.03646946847)
(37.504059366187, 127.047486752713)


In [ ]:
def compare_two_halls(driver, hall_a: str, hall_b: str):
    cypher = """
    MATCH (a:Hall {name:$hall_a})
    MATCH (b:Hall {name:$hall_b})
    RETURN
        a.name AS hallAName,
        a.rating AS hallARating,
        a.reviewCnt AS hallAReviewCnt,
        a.minMealPrice AS hallAMinMealPrice,
        a.maxMealPrice AS hallAMaxMealPrice,
        a.minRentalPrice AS hallAMinRentalPrice,
        a.maxRentalPrice AS hallAMaxRentalPrice,
        b.name AS hallBName,
        b.rating AS hallBRating,
        b.reviewCnt AS hallBReviewCnt,
        b.minMealPrice AS hallBMinMealPrice,
        b.maxMealPrice AS hallBMaxMealPrice,
        b.minRentalPrice AS hallBMinRentalPrice,
        b.maxRentalPrice AS hallBMaxRentalPrice
    """
    with driver.session() as session:
        result = session.run(cypher, hall_a=hall_a, hall_b=hall_b)
        return [record.data() for record in result]

In [ ]:
def extract_multiple_halls(message: str):
    """LLM을 사용해 메시지에서 웨딩홀 이름 추출 (regex 오파싱 방지)"""
    prompt = f"""다음 메시지에서 비교 대상인 웨딩홀/예식장 이름만 추출하세요.
JSON 배열로만 반환하세요 (설명 없이, 마크다운 없이).

메시지: {message}

예시:
입력: "아모리스 역삼점이랑 JW Marriott 서울 이거 두개 비교"
출력: ["아모리스 역삼점", "JW Marriott 서울"]

입력: "노보텔 앰버서더와 삼정호텔 비교해줘"
출력: ["노보텔 앰버서더", "삼정호텔"]

입력: "엘타워, 더채플앳청담, 아펠가모 반포점 비교"
출력: ["엘타워", "더채플앳청담", "아펠가모 반포점"]
"""
    try:
        result = llm.invoke(prompt).content.strip()
        import re as _re
        match = _re.search(r'\[.*?\]', result, _re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception as e:
        print(f"extract_multiple_halls LLM error: {e}")
    return []


In [ ]:
print(extract_multiple_halls("노보텔과 삼정호텔 비교해줘"))
# ['노보텔', '삼정호텔']

print(extract_multiple_halls("노보텔 앰버서더와 삼정호텔 비교해줘"))
# ['노보텔 앰버서더', '삼정호텔']

print(extract_multiple_halls("노보텔, 삼정호텔, 아펠가모 반포점 비교해줘"))
# ['노보텔', '삼정호텔', '아펠가모 반포점']

['노보텔', '삼정호텔']
['노보텔 앰버서더', '삼정호텔']
['노보텔', '삼정호텔', '아펠가모 반포점']


In [ ]:
def resolve_hall_name(driver, keyword: str):
    cypher = """
    MATCH (h:Hall)
    WHERE h.name CONTAINS $keyword
       OR replace(h.name, " ", "") CONTAINS replace($keyword, " ", "")
    RETURN h.name AS name,
           h.rating AS rating,
           h.reviewCnt AS reviewCnt
    ORDER BY h.reviewCnt DESC, h.rating DESC
    LIMIT 1
    """
    with driver.session() as session:
        result = session.run(cypher, keyword=keyword)
        row = result.single()
        return row["name"] if row else None

In [ ]:
def resolve_multiple_hall_names(driver, hall_keywords):
    resolved = []

    for keyword in hall_keywords:
        real_name = resolve_hall_name(driver, keyword)
        if real_name:
            resolved.append(real_name)

    # 중복 제거
    resolved = list(dict.fromkeys(resolved))
    return resolved

In [ ]:
# compare_multiple_halls는 아래 셀(57)에서 정의됨

In [ ]:
def compare_multiple_halls(driver, hall_names):
    cypher = """
    MATCH (h:Hall)
    WHERE h.name IN $hall_names
    RETURN
        h.name AS name,
        h.region AS region,
        h.subRegion AS subRegion,
        h.address AS address,
        h.rating AS rating,
        h.reviewCnt AS reviewCnt,
        h.minMealPrice AS minMealPrice,
        h.maxMealPrice AS maxMealPrice,
        h.minRentalPrice AS minRentalPrice,
        h.maxRentalPrice AS maxRentalPrice,
        h.minIndividualHallPrice AS minHallPrice,
        h.maxIndividualHallPrice AS maxHallPrice,
        h.memoContent AS memoContent,
        h.tel AS tel
    ORDER BY h.reviewCnt DESC, h.rating DESC
    """
    with driver.session() as session:
        result = session.run(cypher, hall_names=hall_names)
        rows = [record.data() for record in result]

    # 같은 이름의 홀이 여러 지역에 있을 경우 리뷰 수 기준 상위 1개만 유지
    seen = {}
    deduped = []
    for row in rows:
        name = row.get("name")
        if name not in seen:
            seen[name] = True
            deduped.append(row)
    return deduped

In [ ]:
def make_multi_compare_answer(rows):
    if not rows:
        return "웨딩홀 비교 결과를 찾지 못했습니다."

    blocks = [f"웨딩홀 비교 결과입니다. (총 {len(rows)}개)\n"]

    for idx, row in enumerate(rows, start=1):
        hall_price_min = row.get("minHallPrice")
        hall_price_max = row.get("maxHallPrice")
        rental_min = row.get("minRentalPrice")
        rental_max = row.get("maxRentalPrice")
        meal_min = row.get("minMealPrice")
        meal_max = row.get("maxMealPrice")
        memo = (row.get("memoContent") or "").strip()

        def fmt_price(v):
            return f"{int(v):,}원" if v is not None else "정보없음"

        price_lines = []
        if hall_price_min is not None:
            price_lines.append(f"  · 홀 전체 대관료: {fmt_price(hall_price_min)} ~ {fmt_price(hall_price_max)}")
        if rental_min is not None:
            price_lines.append(f"  · 예식 렌탈료: {fmt_price(rental_min)} ~ {fmt_price(rental_max)}")
        if meal_min is not None:
            price_lines.append(f"  · 식대: {fmt_price(meal_min)} ~ {fmt_price(meal_max)}")

        price_str = "\n".join(price_lines) if price_lines else "  · 가격 정보없음"

        memo_str = ""
        if memo:
            memo_short = memo[:300] + "..." if len(memo) > 300 else memo
            memo_str = "\n특징: " + memo_short

        block = (
            str(idx) + ". " + row["name"] + "\n"
            + "지역: " + str(row.get("region", "정보없음")) + " " + str(row.get("subRegion", "")) + "\n"
            + "평점: " + str(row.get("rating", "정보없음")) + " / 리뷰수: " + str(row.get("reviewCnt", "정보없음")) + "\n"
            + "가격:\n" + price_str
            + memo_str
        )
        blocks.append(block)

    return "\n\n".join(blocks)

## LLM / Retriever / GraphRAG

In [ ]:
from neo4j_graphrag.generation import RagTemplate
from neo4j_graphrag.retrievers.text2cypher import Text2CypherTemplate

SYSTEM_INSTRUCTIONS = """당신은 웨딩홀 추천 챗봇입니다.
반드시 아래 규칙을 따르세요:
1. Context(데이터베이스 검색 결과)에 있는 웨딩홀을 우선적으로 추천하세요.
2. 사용자 조건을 모두 만족하는 홀이 없더라도, Context에 있는 가장 근접한 결과를 추천하세요.
   예: 강남에 자연채광 홀이 없으면 → 강남 홀 중 가장 평점 높은 것을 추천하면서 "자연채광 조건은 정확히 맞지 않지만..."이라고 안내
3. Context가 완전히 비어있더라도 "없습니다"로 끝내지 마세요. 조건을 완화한 대안을 제안하세요.
4. Context에 없는 웨딩홀을 새로 만들어내거나 추가하지 마세요.
5. 응답 마지막에 항상 추가 조건 조정 여부를 물어보세요.
"""

CYPHER_TEMPLATE = """Task: Generate a Cypher statement for querying a Neo4j graph database from a user input.

위치/특성 검색 규칙 (반드시 따를 것):
- 구 단위 지역(강남구, 서초구 등): MATCH (h:Hall)-[:IN_DISTRICT]->(d:District) WHERE d.name CONTAINS '강남'
- 역/동네 이름(역삼, 홍대 등): h.name CONTAINS '역삼' OR h.address CONTAINS '역삼'
- 광역 지역(서울, 부산 등): h.region CONTAINS '서울'
- 특성/분위기 검색(자연채광, 모던, 야외, 정원 등): h.memoContent CONTAINS '자연채광'
- 조건이 여러 개면 모든 조건 충족 못할 수 있으므로 OR로 넓게 검색하거나 OPTIONAL MATCH 활용
- 홀 이름 검색 시 유사 표기도 함께: h.name CONTAINS '컨티넨탈' OR h.name CONTAINS '컨티넨털' OR h.name CONTAINS 'Continental'

Hall 노드 주요 속성:
- name: 홀 이름
- rating: 평점 (0-100)
- reviewCnt: 리뷰 수
- region: 광역시/도 (예: '서울특별시')
- subRegion: 구 (예: '강남구')
- address: 주소
- memoContent: 홀 상세 설명 (스타일, 예식형태, 식사, 특징 등)
- typeName: 홀 타입

Schema:
{schema}

Examples (optional):
{examples}

Input:
{query_text}

Do not use any properties or relationships not included in the schema.
Do not include triple backticks ``` or any additional text except the generated Cypher statement in your response.

Cypher query:"""

rag_template = RagTemplate(system_instructions=SYSTEM_INSTRUCTIONS)

retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=neo4j_schema,
    examples=examples,
    custom_prompt=CYPHER_TEMPLATE,
)

rag = GraphRAG(
    retriever=retriever,
    llm=llm,
    prompt_template=rag_template,
)


### Retriever / RAG 테스트

In [ ]:
query_text = "서울 강남의 괜찮은 웨딩홀 추천해줘"
search_result = retriever.search(query_text=query_text)

print("=== 생성된 Cypher ===")
print(search_result.metadata.get("cypher", ""))

print("=== 조회 결과 ===")
print(search_result.items)

query_text = "더리버사이드호텔 웨딩과 비슷한 웨딩홀 중 평점이 높은 웨딩홀은 어디인가요?"
response = rag.search(query_text=query_text, return_context=True)

print("=== RAG 생성 Cypher ===")
print(response.retriever_result.metadata.get("cypher", ""))

print("=== 최종 답변 ===")
print(response.answer)

=== 생성된 Cypher ===
MATCH (h:Hall)-[:IN_DISTRICT]->(d:District)
WHERE d.name CONTAINS '강남' AND h.region CONTAINS '서울'
RETURN h.partnerId AS hallId, h.name AS name, h.rating AS rating, h.reviewCnt AS reviewCnt, d.name AS district
ORDER BY h.rating DESC, h.reviewCnt DESC
LIMIT 10
=== 조회 결과 ===
[RetrieverResultItem(content="<Record hallId=11958 name='메리스에이프럴' rating=88.0 reviewCnt=244 district='강남구'>", metadata=None), RetrieverResultItem(content="<Record hallId=11380 name='그래머시 코엑스_강남' rating=86.0 reviewCnt=15 district='강남구'>", metadata=None), RetrieverResultItem(content="<Record hallId=11663 name='더채플앳논현' rating=82.0 reviewCnt=211 district='강남구'>", metadata=None), RetrieverResultItem(content="<Record hallId=8720 name='소노펠리체 컨벤션' rating=82.0 reviewCnt=199 district='강남구'>", metadata=None), RetrieverResultItem(content="<Record hallId=9945 name='노블발렌티 삼성점' rating=82.0 reviewCnt=127 district='강남구'>", metadata=None), RetrieverResultItem(content="<Record hallId=1560 name='상록아트홀_강남' rating=81.0 r

### Gradio

In [ ]:
def safe_str(x, max_len=6000):
    try:
        s = json.dumps(x, ensure_ascii=False, indent=2)
    except Exception:
        s = str(x)
    return s if len(s) <= max_len else s[:max_len] + "\n... (truncated)"


class Seafoam(gr.themes.Base):
    pass


seafoam = Seafoam()

In [ ]:
raw_halls, _ = search_halls_for_location(driver, region="서울", districts=[], limit=10)
print(type(raw_halls))
print(len(raw_halls))
for h in raw_halls[:5]:
    print(h)

<class 'list'>
10
{'hallId': 11180, 'name': '라이즈호텔', 'address': '서울 마포구', 'region': '서울', 'district': '마포구', 'rating': 0.0, 'reviewCnt': 7, 'minHallPrice': 24000000, 'maxHallPrice': 24000000, 'minMealPrice': 95000, 'maxMealPrice': 95000, 'name_match': False}
{'hallId': 11958, 'name': '메리스에이프럴', 'address': '서울 강남구', 'region': '서울', 'district': '강남구', 'rating': 88.0, 'reviewCnt': 244, 'minHallPrice': 15000000, 'maxHallPrice': 15000000, 'minMealPrice': 70000, 'maxMealPrice': 70000, 'name_match': False}
{'hallId': 1512, 'name': '더뉴컨벤션웨딩', 'address': '서울 강서구', 'region': '서울', 'district': '강서구', 'rating': 63.0, 'reviewCnt': 151, 'minHallPrice': 29500000, 'maxHallPrice': 31500000, 'minMealPrice': 82000, 'maxMealPrice': 82000, 'name_match': False}
{'hallId': 13015, 'name': '세인트 메리엘_강남', 'address': '서울', 'region': '서울', 'district': '강남구', 'rating': 0.0, 'reviewCnt': 12, 'minHallPrice': 29400000, 'maxHallPrice': 34300000, 'minMealPrice': 98000, 'maxMealPrice': 98000, 'name_match': False}
{'hallI

In [ ]:
result = recommend_by_location(driver, conn, COUPLE_ID, "역삼역에서 가까운 웨딩홀 추천해줘")
print(result)

{'mode': 'distance', 'start_location': '역삼역', 'region': '서울', 'results': [{'hallId': 10474, 'name': 'JW Marriott 서울_서초', 'address': '서울 서초구', 'region': '서울', 'district': '서초구', 'rating': 82.0, 'reviewCnt': 18, 'minHallPrice': 79200000, 'maxHallPrice': 79200000, 'minMealPrice': 187000, 'maxMealPrice': 187000, 'name_match': False, 'distance_km': 1.93}, {'hallId': 760, 'name': '더리버사이드호텔 웨딩', 'address': '서울 서초구', 'region': '서울', 'district': '서초구', 'rating': 80.0, 'reviewCnt': 688, 'minHallPrice': 21900000, 'maxHallPrice': 37750000, 'minMealPrice': 89000, 'maxMealPrice': 95000, 'name_match': False, 'distance_km': 1.93}, {'hallId': 1237, 'name': '아펠가모 반포점', 'address': '서울 서초구', 'region': '서울', 'district': '서초구', 'rating': 79.0, 'reviewCnt': 245, 'minHallPrice': 40000000, 'maxHallPrice': 44000000, 'minMealPrice': 100000, 'maxMealPrice': 110000, 'name_match': False, 'distance_km': 1.93}, {'hallId': 1197, 'name': 'aT포레웨딩홀', 'address': '서울 서초구', 'region': '서울', 'district': '서초구', 'rating': 76.0,

### Gradio App

In [ ]:
## ===== 웨딩홀 투어 계획 관련 함수 =====

from itertools import permutations
from datetime import datetime, timedelta


def get_kakao_driving_info(origin_lat, origin_lon, dest_lat, dest_lon):
    """카카오 모빌리티 API로 두 지점 간 실제 도로 거리/시간 조회
    Returns: (distance_km, duration_min) or None
    """
    import os, requests
    api_key = os.environ.get("KAKAO_REST_API_KEY")
    if not api_key:
        return None

    url = "https://apis-navi.kakao.com/v1/directions"
    headers = {"Authorization": f"KakaoAK {api_key}"}
    params = {
        "origin": f"{origin_lon},{origin_lat}",
        "destination": f"{dest_lon},{dest_lat}",
        "priority": "RECOMMEND",
        "car_type": 1,
        "summary": "false"
    }

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=5)
        data = resp.json()
        routes = data.get("routes", [])
        if routes and routes[0].get("result_code") == 0:
            summary = routes[0].get("summary", {})
            distance = summary.get("distance", 0) / 1000  # m -> km
            duration = summary.get("duration", 0) / 60    # s -> min
            return round(distance, 2), round(duration, 1)
    except Exception as e:
        print(f"카카오 내비 API 오류: {e}")
    return None


def get_hall_coordinates_for_tour(driver, hall_names):
    """홀 이름 목록에서 좌표 포함한 홀 정보 반환"""
    cypher = """
    MATCH (h:Hall)
    WHERE h.name IN $names
    RETURN h.partnerId AS hallId, h.name AS name,
           h.address AS address, h.region AS region,
           h.subRegion AS subRegion, h.tel AS tel,
           h.rating AS rating, h.reviewCnt AS reviewCnt
    ORDER BY h.reviewCnt DESC
    """
    try:
        with driver.session() as session:
            rows = session.run(cypher, names=hall_names)
            hall_data = [dict(r) for r in rows]
    except Exception as e:
        print(f"Neo4j 홀 조회 오류: {e}")
        return []

    # 같은 이름이 여러 지역에 있을 경우 리뷰 수 기준 상위 1개만 유지
    seen = {}
    deduped = []
    for h in hall_data:
        name = h.get('name')
        if name not in seen:
            seen[name] = True
            deduped.append(h)
    hall_data = deduped

    # 못 찾은 홀은 이름만으로라도 포함
    found_names = {h['name'] for h in hall_data}
    for name in hall_names:
        if name not in found_names:
            hall_data.append({'hallId': None, 'name': name, 'address': None,
                               'region': None, 'subRegion': None, 'tel': None,
                               'rating': None, 'reviewCnt': None})

    results = []
    for hall in hall_data:
        name = hall.get('name') or ''
        address = hall.get('address') or ''
        # 카카오 API 좌표 조회: 홀명 > 주소 순
        coord = geocode_place(f'{name} 웨딩홀')
        if not coord:
            coord = geocode_place(name)
        if not coord and address:
            coord = geocode_place(address)
        hall['lat'] = coord[0] if coord else None
        hall['lon'] = coord[1] if coord else None
        results.append(hall)
    return results


def build_tour_distance_matrix(halls_with_coords, start_coord=None, transport_mode="car"):
    """홀 간 거리/시간 행렬
    - 자가용: 카카오 드라이빙 API (도로거리 + 실시간 교통 반영 시간)
    - 대중교통/도보: 카카오 도로거리 기준 고정 속도 공식
    """
    points = []
    if start_coord:
        points.append({'name': '출발지', 'lat': start_coord[0], 'lon': start_coord[1], 'hallId': None})
    for h in halls_with_coords:
        if h.get('lat') and h.get('lon'):
            points.append(h)

    n = len(points)
    dist_matrix = [[0.0] * n for _ in range(n)]
    time_matrix = [[0.0] * n for _ in range(n)]

    for i in range(n):
        for j in range(i + 1, n):
            p1, p2 = points[i], points[j]
            kakao_result = get_kakao_driving_info(p1['lat'], p1['lon'], p2['lat'], p2['lon'])
            if transport_mode == 'car':
                if kakao_result:
                    dist, time_val = kakao_result  # 카카오 네비 실시간 시간 사용
                else:
                    dist = haversine_distance(p1['lat'], p1['lon'], p2['lat'], p2['lon'])
                    time_val = (dist / 30) * 60  # fallback: 시내 평균 30km/h
            else:
                # 대중교통/도보: 도로거리 기준 고정 공식 (일관된 시간 제공)
                dist = kakao_result[0] if kakao_result else haversine_distance(p1['lat'], p1['lon'], p2['lat'], p2['lon'])
                if transport_mode == 'walk':
                    time_val = (dist / 4) * 60        # 도보 평균 4km/h
                else:  # transit
                    time_val = (dist / 20) * 60 + 5  # 대중교통 평균 20km/h + 환승 5분
            dist_matrix[i][j] = dist_matrix[j][i] = dist
            time_matrix[i][j] = time_matrix[j][i] = time_val

    return points, dist_matrix, time_matrix


def optimize_tour_route(halls_with_coords, start_coord=None, transport_mode="car"):
    """TSP 최단 방문 순서 계산 (≤8홀: 브루트포스, >8홀: 최근접이웃 greedy)
    Returns: dict {optimal_order, total_distance_km, total_travel_min, legs} or None
    """
    valid = [h for h in halls_with_coords if h.get('lat') and h.get('lon')]
    if not valid:
        return None
    if len(valid) == 1:
        return {'optimal_order': valid, 'total_distance_km': 0, 'total_travel_min': 0, 'legs': []}

    points, dist_m, time_m = build_tour_distance_matrix(valid, start_coord, transport_mode)

    def nearest_neighbor_path(start):
        visited = {start}
        path = [start]
        cur = start
        while len(visited) < len(points):
            nxt = min((i for i in range(len(points)) if i not in visited), key=lambda i: dist_m[cur][i])
            path.append(nxt); visited.add(nxt); cur = nxt
        return path

    def path_cost(path):
        return sum(dist_m[path[k]][path[k+1]] for k in range(len(path)-1))

    if start_coord:
        hall_idx = list(range(1, len(points)))
        if len(hall_idx) <= 8:
            best_order = min(([0] + list(p) for p in permutations(hall_idx)), key=path_cost)
        else:
            best_order = nearest_neighbor_path(0)
    else:
        all_idx = list(range(len(points)))
        if len(all_idx) <= 8:
            best_order = min((list(p) for p in permutations(all_idx)), key=path_cost)
        else:
            best_order = nearest_neighbor_path(0)

    legs = []
    total_travel = 0
    for k in range(len(best_order) - 1):
        i, j = best_order[k], best_order[k+1]
        legs.append({'from': points[i].get('name',''), 'to': points[j].get('name',''),
                     'distance_km': round(dist_m[i][j], 1), 'travel_min': round(time_m[i][j])})
        total_travel += time_m[i][j]

    ordered = [points[i] for i in best_order if points[i].get('name') != '출발지']
    return {
        'optimal_order': ordered,
        'total_distance_km': round(path_cost(best_order), 1),
        'total_travel_min': round(total_travel),
        'legs': legs
    }


def format_tour_plan_response(tour_result, start_location=None, visit_duration_min=90, transport_mode="car", buffer_min=10, slot_min=30):
    """투어 계획 결과를 읽기 좋은 텍스트로 포맷"""
    if not tour_result:
        return '투어 계획 생성 실패: 홀 위치 정보를 찾을 수 없어요.'

    ordered = tour_result['optimal_order']
    legs = tour_result['legs']
    total_dist = tour_result['total_distance_km']
    total_travel = tour_result['total_travel_min']
    def ceil_to_slot(dt, minutes=30):
        remainder = dt.minute % minutes
        if remainder == 0 and dt.second == 0 and dt.microsecond == 0:
            return dt.replace(second=0, microsecond=0)
        delta = minutes - remainder
        return (dt + timedelta(minutes=delta)).replace(second=0, microsecond=0)

    lines = []
    lines.append('## 🗺️ 웨딩홀 투어 최단경로 추천 플랜')
    lines.append('')
    if start_location:
        lines.append(f'📍 **출발지**: {start_location}')
    lines.append('최단 이동 경로를 기준으로, 상담 지연을 고려해 **30분 단위로 여유 있게** 일정을 잡았어요.')
    lines.append(f'🏛️ 방문 홀 **{len(ordered)}개** | 📊 총 이동거리 **약 {total_dist}km** | 🚗 순수 이동시간 **약 {total_travel}분**')
    lines.append('')
    lines.append('---')
    lines.append('### 📋 추천 방문 순서 (오전 10:00 출발 기준)')
    lines.append('')

    current_time = datetime.strptime('10:00', '%H:%M')
    start_dt = current_time

    for idx, hall in enumerate(ordered):
        name = hall.get('name', '')
        address = hall.get('address') or hall.get('subRegion') or hall.get('region', '')
        rating = hall.get('rating')
        review = hall.get('reviewCnt')

        arrive_str = current_time.strftime('%H:%M')
        leave_time = current_time + timedelta(minutes=visit_duration_min)
        leave_str = leave_time.strftime('%H:%M')

        lines.append(f'**{idx + 1}. {name}**')
        if address:
            lines.append(f'   📍 {address}')
        if rating is not None:
            lines.append(f'   ⭐ 평점 {rating} | 리뷰 {review or 0}건')
        lines.append(f'   🕐 방문 시간: **{arrive_str} ~ {leave_str}** (약 {visit_duration_min}분)')

        current_time = leave_time
        if idx < len(legs):
            leg = legs[idx]
            lines.append(f'   ⬇️ 이동: 약 {leg["distance_km"]}km · 약 {leg["travel_min"]}분 (버퍼 {buffer_min}분 포함 전 다음 일정 준비)')
            current_time += timedelta(minutes=leg['travel_min'] + buffer_min)
            current_time = ceil_to_slot(current_time, slot_min)
        lines.append('')

    end_str = current_time.strftime('%H:%M')
    total_time = int((current_time - start_dt).total_seconds() // 60)
    lines.append('---')
    lines.append(f'⏰ **예상 종료**: 약 {end_str}  (총 약 {total_time}분 소요)')
    lines.append('')
    lines.append('### 💡 투어 팁')
    lines.append('- **평일 오전~오후** 방문 시 가장 여유롭습니다')
    lines.append('- 홀당 약 1.5~2시간 여유 있게 준비하세요')
    lines.append('- 가격·식사·수용인원 비교표를 미리 출력해 가세요')

    return '\n'.join(lines)


In [ ]:
from openai import OpenAI as _OpenAI
_oai = _OpenAI()

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_near_location",
            "description": "특정 위치/역 근처의 웨딩홀을 거리순으로 검색합니다. 위치 기반 검색에 사용.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "검색 기준 위치 (예: 강남역, 홍대, 서초동)"},
                    "count": {"type": "integer", "description": "추천 개수 (기본값 5)"},
                    "sort_by": {"type": "string", "enum": ["distance", "rating", "price_asc", "price_desc", "review"]},
                    "max_price": {"type": "integer", "description": "최대 홀 전체 대관료 (원)"},
                    "min_price": {"type": "integer", "description": "최소 홀 전체 대관료 (원)"},
                    "features": {"type": "array", "items": {"type": "string"}, "description": "시설 조건 (주차, 야외, 채플 등)"},
                    "mood": {"type": "array", "items": {"type": "string"}, "description": "분위기 (럭셔리, 로맨틱, 모던 등)"}
                },
                "required": ["location"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_by_features",
            "description": "특성/스타일/분위기/가격 조건으로 웨딩홀을 검색합니다. 위치 무관 조건 검색.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "검색 조건을 담은 자연어 쿼리"},
                    "count": {"type": "integer", "description": "결과 개수 (기본값 5)"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "compare_halls",
            "description": "웨딩홀들의 상세 정보(가격, 평점, 특징)를 비교합니다. 대화 맥락에서 홀 이름을 직접 파악해서 전달하세요.",
            "parameters": {
                "type": "object",
                "properties": {
                    "hall_names": {"type": "array", "items": {"type": "string"}, "description": "비교할 웨딩홀 이름 목록"},
                    "sort_by": {"type": "string", "enum": ["rating", "price_asc", "price_desc", "review"]},
                    "count": {"type": "integer", "description": "상위 N개만 비교"}
                },
                "required": ["hall_names"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_hall_details",
            "description": "특정 웨딩홀의 상세 정보(대관료, 식대, 수용인원, 주소, 주차, 예식형태 등)를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "hall_name": {"type": "string", "description": "웨딩홀 이름"}
                },
                "required": ["hall_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_my_preferences",
            "description": "사용자의 저장된 선호 정보(지역, 스타일, 분위기, 예산 등)를 조회합니다.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "recommend_by_my_preferences",
            "description": "저장된 사용자 선호 기반으로 웨딩홀을 추천합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "count": {"type": "integer", "description": "추천 개수 (기본값 5)"}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "plan_tour_route",
            "description": "여러 웨딩홀을 효율적으로 방문하는 최단경로/투어 일정을 계획합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "hall_names": {"type": "array", "items": {"type": "string"}, "description": "방문할 웨딩홀 이름 목록"},
                    "start_location": {"type": "string", "description": "출발지"},
                    "transport": {"type": "string", "enum": ["car", "transit", "walk"], "description": "이동수단"}
                },
                "required": ["hall_names", "start_location"]
            }
        }
    }
]

TOOL_SYSTEM_PROMPT = """당신은 웨딩홀 추천 전문 챗봇입니다.
사용자의 질문을 정확히 이해하고, 필요한 도구를 호출해 실제 데이터 기반으로 답변하세요.

도구 선택 기준:
- 위치/역 근처 검색 -> search_near_location
- 스타일/분위기/가격 조건 검색 -> search_by_features
- 홀 비교 (이것들, 위에 것들, 특정 홀 이름) -> compare_halls (대화에서 홀 이름 직접 추출)
- 특정 홀 상세 정보 -> get_hall_details
- 내 취향/선호 -> get_my_preferences 또는 recommend_by_my_preferences
- 투어 경로/동선 -> plan_tour_route

투어 경로 특별 규칙 (매우 중요):
- plan_tour_route 호출 전에 반드시 아래 두 가지를 사용자에게 먼저 물어보세요:
  1. 출발지 (예: 강남역, 판교역, 집 주소 등) - 절대 추측하거나 생략하지 말 것
  2. 이동수단: 자가용 / 대중교통 / 도보 중 선택 (명시 없으면 반드시 물어볼 것)
- 두 정보가 모두 확보된 경우에만 plan_tour_route를 호출하세요
- 질문 예시: "출발지와 이동수단을 알려주세요! (예: 강남역, 자가용)"

응답 원칙:
- 도구 결과 데이터만 기반으로 답변 (없는 정보 생성 금지)
- 한국어로 친절하게 답변
- 답변 후 자연스러운 추가 질문 포함
"""


def _tool_search_near_location(location, count=5, sort_by="distance", max_price=None, min_price=None, features=None, mood=None):
    result = recommend_by_location(driver, conn, COUPLE_ID, location, start_location=location, count=max(count * 3, 20))
    if result.get("error"):
        return {"error": result["error"]}
    halls = result.get("results", [])
    if max_price:
        halls = [h for h in halls if (h.get("minHallPrice") or 999999999) <= max_price]
    if min_price:
        halls = [h for h in halls if (h.get("minHallPrice") or 0) >= min_price]
    if features:
        _FEAT_SYN = {"주차": ["주차", "발렛"], "야외": ["야외", "가든", "루프탑", "테라스"], "채플": ["채플"], "버진로드": ["버진"]}
        def _has_feat(h):
            memo = (h.get("memoContent") or "") + (h.get("name") or "")
            return all(any(s in memo for s in _FEAT_SYN.get(f, [f])) for f in features)
        halls = [h for h in halls if _has_feat(h)]
    sort_map = {
        "rating": lambda x: -(x.get("rating") or 0),
        "price_asc": lambda x: (x.get("minHallPrice") or 99999999),
        "price_desc": lambda x: -(x.get("minHallPrice") or 0),
        "review": lambda x: -(x.get("reviewCnt") or 0),
    }
    if sort_by and sort_by != "distance" and sort_by in sort_map:
        halls = sorted(halls, key=sort_map[sort_by])
    halls = halls[:count]
    return {
        "location": result.get("start_location", location),
        "sort_by": sort_by,
        "halls": [{"name": h.get("name"), "address": h.get("address"), "distance_km": h.get("distance_km"), "rating": h.get("rating"), "reviewCnt": h.get("reviewCnt"), "minHallPrice": h.get("minHallPrice")} for h in halls],
        "neo4j_query": result.get("neo4j_query", "")
    }


def _tool_search_by_features(query, count=5):
    rag_result = rag.search(query_text=query, return_context=True)
    cypher = rag_result.retriever_result.metadata.get("cypher", "")
    items = rag_result.retriever_result.items
    return {"answer": rag_result.answer, "halls": [str(i) for i in (items or [])], "cypher": cypher}


def _tool_compare_halls(hall_names, sort_by=None, count=None):
    resolved = resolve_multiple_hall_names(driver, hall_names)
    if len(resolved) < 2:
        return {"error": f"홀을 2개 이상 찾지 못했습니다. 찾은 홀: {resolved}"}
    rows = compare_multiple_halls(driver, resolved)
    sort_map = {
        "rating": lambda x: -(x.get("rating") or 0),
        "price_asc": lambda x: (x.get("minHallPrice") or 99999999),
        "price_desc": lambda x: -(x.get("minHallPrice") or 0),
        "review": lambda x: -(x.get("reviewCnt") or 0),
    }
    if sort_by and sort_by in sort_map:
        rows = sorted(rows, key=sort_map[sort_by])
    if count and count < len(rows):
        rows = rows[:count]
    return {"halls": rows}


def _tool_get_hall_details(hall_name):
    cypher = """MATCH (h:Hall) WHERE h.name CONTAINS $name
OPTIONAL MATCH (h)-[:HAS_STYLE_FILTER]->(sf:StyleFilter)
WITH h, collect(sf.name) AS styles
RETURN h.name AS name, h.minIndividualHallPrice AS minHallPrice, h.maxIndividualHallPrice AS maxHallPrice,
       h.minRentalPrice AS minRentalPrice, h.maxRentalPrice AS maxRentalPrice,
       h.minMealPrice AS minMealPrice, h.maxMealPrice AS maxMealPrice,
       h.address AS address, h.region AS region, h.subRegion AS district,
       h.tel AS tel, h.rating AS rating, h.reviewCnt AS reviewCnt,
       h.memoContent AS memoContent, styles
ORDER BY h.reviewCnt DESC LIMIT 3"""
    with driver.session() as session:
        rows = [dict(r) for r in session.run(cypher, name=hall_name)]
    return {"halls": rows} if rows else {"error": f"'{hall_name}' 홀을 찾지 못했습니다."}


def _tool_get_my_preferences():
    pref = get_user_preference(conn, COUPLE_ID)
    return pref if pref else {"error": "저장된 선호 정보가 없습니다."}


def _tool_recommend_by_my_preferences(count=5):
    pref = get_user_preference(conn, COUPLE_ID)
    if not pref:
        return {"error": "저장된 선호 정보가 없습니다."}
    halls = recommend_by_preference(driver, pref)
    return {"halls": halls[:count]}


def _tool_plan_tour_route(hall_names, start_location, transport="car"):
    start_coord = geocode_place(start_location) if start_location else None
    halls_with_coords = get_hall_coordinates_for_tour(driver, hall_names)
    valid_halls = [h for h in halls_with_coords if h.get('lat') and h.get('lon')]
    if not valid_halls:
        return {"plan": "홀의 위치 정보를 찾을 수 없어요. 홀 이름을 다시 확인해 주세요."}
    tour_result = optimize_tour_route(valid_halls, start_coord=start_coord, transport_mode=transport)
    result_text = format_tour_plan_response(tour_result, start_location=start_location, transport_mode=transport)
    return {"plan": result_text}


_TOOL_HANDLERS = {
    "search_near_location": lambda args: _tool_search_near_location(**{k: v for k, v in args.items() if v is not None}),
    "search_by_features": lambda args: _tool_search_by_features(**{k: v for k, v in args.items() if v is not None}),
    "compare_halls": lambda args: _tool_compare_halls(**{k: v for k, v in args.items() if v is not None}),
    "get_hall_details": lambda args: _tool_get_hall_details(**{k: v for k, v in args.items() if v is not None}),
    "get_my_preferences": lambda args: _tool_get_my_preferences(),
    "recommend_by_my_preferences": lambda args: _tool_recommend_by_my_preferences(**{k: v for k, v in args.items() if v is not None}),
    "plan_tour_route": lambda args: _tool_plan_tour_route(**{k: v for k, v in args.items() if v is not None}),
}

In [ ]:
with gr.Blocks(theme=seafoam) as demo:

    def save_message_to_db(couple_id, role, content, intent=None):
        """대화 메시지를 MySQL에 저장"""
        try:
            c = conn.cursor()
            c.execute(
                "INSERT INTO chat_history (couple_id, role, content, intent) VALUES (%s, %s, %s, %s)",
                (couple_id, role, content[:4000], intent)
            )
            conn.commit()
            c.close()
        except Exception as e:
            print(f"DB 저장 오류: {e}")

    def load_history_from_db(couple_id, limit=20):
        """MySQL에서 최근 대화 기록 불러오기"""
        try:
            c = conn.cursor()
            c.execute(
                """SELECT role, content FROM chat_history
                   WHERE couple_id = %s
                   ORDER BY created_at DESC LIMIT %s""",
                (couple_id, limit)
            )
            rows = c.fetchall()
            c.close()
            # 최신순으로 가져왔으니 역순으로
            return [{"role": r, "content": c} for r, c in reversed(rows)]
        except Exception as e:
            print(f"DB 불러오기 오류: {e}")
            return []

    def build_context_from_db_history(couple_id, max_turns=6):
        """DB 기록을 LLM 프롬프트용 문자열로 변환"""
        history = load_history_from_db(couple_id, limit=max_turns * 2)
        if not history:
            return ""
        lines = []
        for m in history:
            role = "사용자" if m["role"] == "user" else "어시스턴트"
            lines.append(f"{role}: {m['content']}")
        return "\n".join(lines)

    def format_history_for_prompt(chat_history, max_turns=5):
        """최근 N턴의 대화 기록을 프롬프트용 문자열로 변환"""
        if not chat_history:
            return ""
        recent = chat_history[-(max_turns * 2):]
        lines = []
        for m in recent:
            role = "사용자" if m["role"] == "user" else "어시스턴트"
            lines.append(f"{role}: {m['content']}")
        return "\n".join(lines)

    def extract_hall_names_from_history(chat_history, max_turns=5, user_message=None):
        """최근 대화(사용자+봇 쌍)를 모두 넘겨 LLM이 맥락을 정확히 파악하게 함.
        - 어시스턴트 메시지만 보면 "강남 추천" vs "사당 추천" 구분 불가
        - 사용자 질문+봇 답변 쌍을 함께 보여줘야 "강남 근처 추천 위에 두개" 같은
          참조를 올바르게 해석할 수 있음
        """
        if not chat_history:
            return []

        # 최근 max_turns 쌍(사용자+봇)을 시간순으로 구성
        recent = chat_history[-(max_turns * 2):]
        conv_lines = []
        for m in recent:
            role = "사용자" if m["role"] == "user" else "봇"
            conv_lines.append(f"[{role}] {m['content']}")
        conv_text = "\n\n".join(conv_lines)

        user_section = f"\n\n[현재 사용자 요청]\n{user_message}" if user_message else ""

        prompt = f"""당신은 대화 맥락을 분석하는 전문가입니다.
{conv_text}{user_section}

위 대화를 읽고, [현재 사용자 요청]이 가리키는 웨딩홀이 무엇인지 판단하세요.

판단 기준:
- 이것들/위에 것들/방금 추천/그것들 등 -> 사용자 요청 바로 직전 봇 답변에서 추천한 홀들
- 강남 추천한거/아까 판교역 꺼 등 특정 이전 추천 명시 -> 해당 봇 답변에서 추천한 홀들
- 홀 이름이 직접 언급된 경우 -> 그 이름들

주의:
- 여러 턴에 걸쳐 다른 지역 홀들이 나왔다면, 사용자가 명시적으로 과거 것을 언급하지 않는 한 가장 최근 추천만 사용
- 비교/투어 결과로 나온 홀 목록은 추천이 아니므로 참조 대상에서 제외
- 중복 제거
- JSON 배열만 출력

출력 예시: ["더에스비웨딩컨벤션", "마리앤코 동작점", "그레이스파티"]
"""
        try:
            result = llm.invoke(prompt).content.strip()
            import re as _re
            match = _re.search(r'\[.*?\]', result, _re.DOTALL)
            if match:
                return json.loads(match.group())
        except Exception:
            pass
        return []

    def parse_intent(message: str, chat_history=None) -> dict:
        """LLM으로 의도와 조건을 한 번에 파싱"""
        history_text = format_history_for_prompt(chat_history, max_turns=10) if chat_history else ""
        history_section = f"[이전 대화]\n{history_text}\n\n" if history_text else ""

        prompt = f"""{history_section}[현재 사용자 메시지]
{message}

위 대화를 분석해서 아래 JSON 형식으로만 응답하세요. 설명 없이 JSON만 출력하세요.

{{
  "intent": "LOCATION | NEO4J_RAG | COMPARE | FOLLOWUP | HALL_INFO | PREFERENCE_RECOMMEND | PREFERENCE_QUERY | LIKE_QUERY | TOUR_PLAN | GENERAL",
  "location": "역/지역 이름 또는 null",
  "max_hall_price": "최대 홀 전체 대관료 숫자(원) 또는 null",
  "min_hall_price": "최소 홀 전체 대관료 숫자(원) 또는 null",
  "features": ["주차", "지하철", "버진로드", "야외" 등 시설 조건 배열],
  "mood": ["럭셔리", "로맨틱", "모던" 등 분위기 배열],
  "sort_by": "distance | price_asc | price_desc | rating | review | null",
  "compare_halls": ["비교할 홀 이름1", "홀 이름2"],
  "is_followup": true 또는 false,
  "followup_filter": "이전 결과에서 필터/정렬할 조건 설명 또는 null",
  "count": "사용자가 요청한 추천 개수 숫자(예: 15) 또는 null"
}}

intent 선택 기준:
- HALL_INFO: 특정 홀 1개의 상세정보 조회 (대관료, 식대, 수용인원, 주소, 주차, 예식형태 등)
- LOCATION: 특정 역/위치 근처 검색 (근처, 가까운, 옆에, 역에서 등)
- NEO4J_RAG: 특성/스타일/분위기/가격으로 웨딩홀 검색
- COMPARE: 두 개 이상 홀 비교
- FOLLOWUP: 이전 추천 결과에서 추가 조건으로 필터 (이중에서, 그중에서, 주차되는곳, 더 싼곳 등)
- PREFERENCE_RECOMMEND: 내 취향/선호 기반 추천
- PREFERENCE_QUERY: 내 선호 설정 조회
- LIKE_QUERY: 찜 목록 조회
- TOUR_PLAN: 웨딩홀 투어/드레스투어 일정·동선·방문순서·최단경로 계획 요청 ("투어", "동선", "어디부터", "순서", "경로" 등이 포함된 경우)
- GENERAL: 그 외

주의: 이전 추천 결과에 추가 조건을 묻는 경우(주차 잘되는곳, 이중에서 싼곳 등)는 반드시 FOLLOWUP으로 분류하세요.
"""
        try:
            result = llm.invoke(prompt).content.strip()
            import re as _re
            # JSON 블록 추출
            match = _re.search(r'\{.*\}', result, _re.DOTALL)
            if match:
                parsed = json.loads(match.group())
                return parsed
        except Exception as e:
            print(f"parse_intent error: {e}")

        # fallback: 키워드 기반
        msg = message.strip()
        hall_info_kw = ["대관료", "식대", "수용인원", "보증인원", "주소", "연락처", "전화", "예식형태", "식사형태"]
        if any(x in msg for x in hall_info_kw):
            return {"intent": "HALL_INFO", "is_followup": False, "features": [], "mood": []}
        if any(x in msg for x in ["근처", "가까운", "옆에", "이내"]):
            return {"intent": "LOCATION", "location": None, "features": [], "mood": [],
                    "max_hall_price": None, "sort_by": "distance", "is_followup": False}
        if any(x in msg for x in ["비교"]):
            return {"intent": "COMPARE", "compare_halls": [], "is_followup": False}
        tour_kw = ["투어", "동선", "방문 순서", "방문순서", "경로", "어디부터", "순서대로", "일정 짜", "일정짜"]
        if any(x in msg for x in tour_kw):
            return {"intent": "TOUR_PLAN", "is_followup": False, "features": [], "mood": []}
        return {"intent": "NEO4J_RAG", "is_followup": False, "features": [], "mood": []}

    def classify_intent(message: str, chat_history=None) -> str:
        """하위 호환용 - parse_intent의 intent 필드만 반환"""
        return parse_intent(message, chat_history).get("intent", "GENERAL")

    TOUR_AWAITING_LOCATION_MARKER = "출발지와 이동수단을 함께 알려주세요"
    TOUR_AWAITING_MARKERS = [
        "출발지와 이동수단을 함께 알려주세요",
        "출발지를 알려주세요",
        "이동수단을 알려주세요",
    ]

    def is_tour_prompt_message(text):
        return any(marker in (text or '') for marker in TOUR_AWAITING_MARKERS)

    def extract_tour_start_location(message):
        cleaned_message = (message or '').strip()
        cleaned_message = re.sub(r'(자가용|대중교통|지하철|버스|도보|차량|자동차|차)$', '', cleaned_message).strip()
        patterns = [
            r"(.+?)에서\s*출발",
            r"출발지(?:는|가)?\s*(.+)",
            r"(.+?)\s*출발",
        ]
        for pattern in patterns:
            m = re.search(pattern, cleaned_message)
            if m:
                location = m.group(1).strip()
                location = re.sub(r"(자가용|대중교통|지하철|버스|도보|차|차량|자동차)$", "", location).strip()
                if len(location) >= 2:
                    return location
        if cleaned_message and len(cleaned_message) <= 20 and not any(x in cleaned_message for x in ["투어", "동선", "웨딩홀", "추천", "근처", "가까운"]):
            if re.search(r'(역|구|동|시|군|읍|면|로|가)$', cleaned_message) or '집' in cleaned_message or '회사' in cleaned_message:
                return cleaned_message
        return None

    def detect_transport_mode(message):
        if any(x in message for x in ["대중교통", "버스", "지하철", "전철", "전동차"]):
            return "transit"
        if any(x in message for x in ["걸어", "도보", "워킹", "걷기"]):
            return "walk"
        if any(x in message for x in ["자가용", "차", "차량", "자동차", "운전"]):
            return "car"
        return None

    def recover_tour_context(chat_history, start_location=None, transport_mode=None):
        recovered_location = start_location
        recovered_transport = transport_mode
        pending_start_idx = None

        for idx in range(len(chat_history) - 1, -1, -1):
            msg = chat_history[idx]
            if msg.get('role') != 'assistant':
                continue
            if is_tour_prompt_message(msg.get('content', '')):
                pending_start_idx = idx
                continue
            if pending_start_idx is not None:
                break

        if pending_start_idx is None:
            return recovered_location, recovered_transport

        for m in reversed(chat_history[pending_start_idx + 1:]):
            if m.get('role') != 'user':
                continue
            text = m.get('content', '')
            if not recovered_location:
                recovered_location = extract_tour_start_location(text)
            if not recovered_transport:
                recovered_transport = detect_transport_mode(text)
            if recovered_location and recovered_transport:
                break
        return recovered_location, recovered_transport

    def handle_tour_plan(message, chat_history, start_location=None, transport_mode=None):
        import re as _re
        # 1. 홀 이름 추출
        hall_names = extract_hall_names_from_history(chat_history, user_message=message)
        if not hall_names:
            hall_names = extract_multiple_halls(message)
        if not hall_names:
            return "투어할 웨딩홀을 먼저 추천받으시거나, 방문하고 싶은 홀 이름을 알려주세요!"

        # 2. "위에 세개", "앞에 3개" 등 숫자 지정 처리
        kor_map = {"하나": 1, "한": 1, "두": 2, "세": 3, "네": 4, "다섯": 5, "여섯": 6}
        count = None
        m_num = _re.search(r"(\d+)\s*개", message)
        if m_num:
            count = int(m_num.group(1))
        else:
            for kor, num in kor_map.items():
                if kor + "개" in message or kor + " 개" in message:
                    count = num
                    break
        if count and count < len(hall_names):
            hall_names = hall_names[:count]

        # 3. 출발지/이동수단이 없으면 반드시 먼저 질문
        if not start_location and not transport_mode:
            halls_str = ", ".join(hall_names)
            ask = "투어할 홀은 **" + halls_str + "** 맞죠?\n\n"
            ask += "출발지와 이동수단을 함께 알려주세요!\n"
            ask += "- **자가용**: '강남역 자가용' 또는 '강남역 차'\n"
            ask += "- **대중교통**: '강남역 대중교통' 또는 '강남역 지하철'\n"
            ask += "- **도보**: '강남역 도보'\n"
            return ask
        if not start_location:
            return "출발지를 알려주세요! 예: 강남역, 판교역, 집 주소"
        if not transport_mode:
            return "이동수단을 알려주세요! 자가용 / 대중교통 / 도보 중에서 선택해 주세요."

        # 4. 좌표 조회 및 경로 최적화
        start_coord = geocode_place(start_location) if start_location else None
        halls_with_coords = get_hall_coordinates_for_tour(driver, hall_names)
        valid_halls = [h for h in halls_with_coords if h.get("lat") and h.get("lon")]
        if not valid_halls:
            return "죄송해요, 홀의 위치 정보를 찾을 수 없었습니다."

        tour_result = optimize_tour_route(halls_with_coords, start_coord, transport_mode)
        return format_tour_plan_response(tour_result, start_location, transport_mode=transport_mode)
    def default_llm(message: str, chat_history=None) -> str:
        history_text = format_history_for_prompt(chat_history) if chat_history else ""
        history_section = f"\n[이전 대화]\n{history_text}\n" if history_text else ""

        prompt_text = f"""당신은 웨딩홀 추천 전문 챗봇입니다. 친절하고 전문적으로 답변해주세요.
{history_section}
[답변 가이드라인]
1. 사용자가 특정 홀, 트렌드, 비교하기를 물어볼 경우 구체적으로 알려주세요.
2. 가격대, 스타일, 위치, 리뷰, 특색 도 함께 알려주세요.
3. 답변 이후 자연스럽게 추가 질문을 해주세요.

user_input: {message}
"""
        return llm.invoke(prompt_text).content

    def response_fn(message, chat_history):
        chat_history = chat_history or []
        _plog = []
        cypher_out = ""
        items_out = ""

        intent_info = parse_intent(message, chat_history)
        last_assistant = chat_history[-1]["content"] if chat_history and chat_history[-1]["role"] == "assistant" else ""
        start_location = extract_tour_start_location(message)
        transport_mode = detect_transport_mode(message)
        waiting_for_tour_inputs = is_tour_prompt_message(last_assistant)
        is_tour_followup = waiting_for_tour_inputs and bool(start_location or transport_mode)
        if intent_info.get("intent") == "TOUR_PLAN" or is_tour_followup:
            if waiting_for_tour_inputs:
                start_location, transport_mode = recover_tour_context(chat_history, start_location, transport_mode)
            answer = handle_tour_plan(message, chat_history, start_location=start_location, transport_mode=transport_mode)
            chat_history.append({"role": "user", "content": message})
            chat_history.append({"role": "assistant", "content": answer})
            save_message_to_db(COUPLE_ID, "user", message, "TOUR_PLAN")
            save_message_to_db(COUPLE_ID, "assistant", answer, "TOUR_PLAN")
            process_text = "[투어 플랜 핸들러 직접 처리]"
            return chat_history, cypher_out, items_out, process_text

        oai_messages = [{"role": "system", "content": TOOL_SYSTEM_PROMPT}]
        for m in chat_history[-20:]:
            oai_messages.append({"role": m["role"], "content": m["content"]})
        oai_messages.append({"role": "user", "content": message})

        answer = ""
        for _iter in range(6):
            response = _oai.chat.completions.create(
                model="gpt-4o",
                messages=oai_messages,
                tools=TOOLS,
                tool_choice="auto",
                temperature=0
            )
            choice = response.choices[0]
            resp_msg = choice.message

            if choice.finish_reason == "stop" or not resp_msg.tool_calls:
                answer = resp_msg.content or ""
                break

            tool_calls_data = [
                {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in resp_msg.tool_calls
            ]
            oai_messages.append({"role": "assistant", "content": resp_msg.content, "tool_calls": tool_calls_data})

            for tc in resp_msg.tool_calls:
                tool_name = tc.function.name
                try:
                    args = json.loads(tc.function.arguments)
                except Exception:
                    args = {}

                _plog.append(f"[도구 호출] {tool_name}({json.dumps(args, ensure_ascii=False)})")

                try:
                    result = _TOOL_HANDLERS[tool_name](args)
                except Exception as e:
                    result = {"error": str(e)}

                result_str = json.dumps(result, ensure_ascii=False)
                if isinstance(result, dict):
                    cypher_out = result.get("neo4j_query", result.get("cypher", cypher_out))
                    items_out = result_str

                _plog.append(f"    결과: {result_str[:300]}{'...' if len(result_str) > 300 else ''}")

                oai_messages.append({"role": "tool", "tool_call_id": tc.id, "content": result_str})

        if not answer:
            answer = "죄송합니다. 처리 중 문제가 발생했습니다."

        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": answer})
        save_message_to_db(COUPLE_ID, "user", message, "TOOL_USE")
        save_message_to_db(COUPLE_ID, "assistant", answer, "TOOL_USE")

        process_text = "\n".join(_plog) if _plog else ""
        return chat_history, cypher_out, items_out, process_text

    with gr.Row():
        gr.HTML("""
        <div style="text-align:center; max-width:1000px; margin:10px auto;">
            <h1>Graph RAG 웨딩홀 챗봇</h1>
            <p>Neo4j 그래프 DB 기반으로 웨딩홀을 추천하고, 생성된 Cypher와 조회 결과를 함께 보여드립니다.</p>
        </div>
        """)

    with gr.Row():
        with gr.Column(scale=1):
            process_log_box = gr.Textbox(label="처리 과정", lines=25, max_lines=60, autoscroll=False)
            generated_query = gr.Textbox(label="생성된 Cypher 쿼리", lines=10)
            query_result = gr.Textbox(label="원본 조회 결과", lines=10)

        with gr.Column(scale=3):
            chatbot = gr.Chatbot(type="messages")
            msg = gr.Textbox(
                placeholder="예: 강남 근처에서 접근성 좋은 웨딩홀 추천해줘",
                label="입력"
            )

            gr.Examples(
                examples=[
                    "웨딩홀 그랜드볼룸 자연채광이 있는 모던한 분위기의 웨딩홀을 추천해줄 수 있나요?",
                    "웨딩홀 라온파라디아회화나무 혹은 채플 스타일 웨딩홀을 추천해주세요.",
                    "서울 강남지역에서 접근성 좋고 리뷰가 많은 웨딩홀 5개 추천해줘",
                    "내 취향에 맞는 웨딩홀 알려줘",
                    "현재 찜한 웨딩홀 목록",
                    "찜한 기반 웨딩홀을 찾아 줘 추천해줘",
                    "강남역에서 가까운 웨딩홀 추천해줘"
                ],
                inputs=[msg]
            )

            with gr.Row():
                btn = gr.Button("Submit", variant="primary")
                clear = gr.Button("Clear")

    submit_event = dict(
        fn=response_fn,
        inputs=[msg, chatbot],
        outputs=[chatbot, generated_query, query_result, process_log_box]
    )

    btn.click(**submit_event).then(lambda: "", None, msg)
    msg.submit(**submit_event).then(lambda: "", None, msg)

    def clear_all():
        return [], "", "", ""

    clear.click(
        fn=clear_all,
        inputs=None,
        outputs=[chatbot, generated_query, query_result, process_log_box],
        queue=False
    ).then(lambda: "", None, msg)

    def load_initial_history():
        return load_history_from_db(COUPLE_ID, limit=40)

    demo.load(fn=load_initial_history, outputs=[chatbot])

demo.launch(debug=True, share=True)


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_16816\1094305819.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=seafoam) as demo:
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_16816\1094305819.py:617: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(type="messages")


* Running on local URL:  http://127.0.0.1:7860
